In [ ]:
# =============================
# ES1 — WordNet + Embeddings
# =============================

!pip -q install gensim nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 61.2 MB/s eta 0:00:00


In [ ]:
import re
from typing import List, Optional, Tuple

import numpy as np
from numpy.linalg import norm
import gensim.downloader as api
from gensim.models import KeyedVectors

import nltk

nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("punkt")
nltk.download("stopwords")
nltk.download('punkt_tab')

from nltk.corpus import wordnet as wn
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.corpus.reader.wordnet import Synset

# -----------------------------
# 0) Setup NLTK (verifica risorse)
# -----------------------------
def ensure_nltk():
    resources = [
        ("tokenizers/punkt", "punkt"),
        ("corpora/wordnet", "wordnet"),
        ("corpora/omw-1.4", "omw-1.4"),
        ("corpora/stopwords", "stopwords"),
    ]
    for path, name in resources:
        try:
            nltk.data.find(path)
        except LookupError:
            nltk.download(name)

# -----------------------------
# 1) Utility base
# -----------------------------

# calcola la similarità coseno tra due vettori
def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    na, nb = norm(a), norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

# pulisce il token: minuscolo e rimuove caratteri speciali
def normalize_token(t: str) -> str:
    t = t.lower()
    t = re.sub(r"[^a-z0-9_']+", "", t)
    return t

# calcola il vettore medio da una lista di vettori
def mean_vec(vs: List[np.ndarray], dim: int) -> np.ndarray:
    if not vs:
        return np.zeros(dim, dtype=np.float32)
    return np.mean(np.stack(vs, axis=0), axis=0)

# -----------------------------
# 2) Carica vettori pre-addestrati
# -----------------------------
def load_pretrained_vectors(name: str = "fasttext-wiki-news-subwords-300") -> KeyedVectors:
    """
    Carica vettori pre-addestrati tramite gensim-data.
    """
    print(f"Caricamento del modello {name} in corso...")
    wv = api.load(name)
    print(f"✅ Loaded pretrained vectors: {name}")
    print("Vocab size:", len(wv))
    print("Vector dim:", wv.vector_size)
    return wv

# -----------------------------
# 3) Synset embedding (WordNet -> vettore)
#    per ogni synset Wordnet costruisce un vettore usando gli embeddings
# -----------------------------
def synset_embedding(
    syn: Synset,
    wv: KeyedVectors,
    use_gloss: bool = True,
    gloss_weight: float = 0.6,
    use_examples: bool = True,
    examples_weight: float = 0.4,
) -> np.ndarray:
    dim = wv.vector_size

    # 1. Vettori dei lemmi (nome del concetto)
    lemma_vecs = []
    for lemma in syn.lemmas():
        # lemma multiword
        parts = [normalize_token(p) for p in lemma.name().replace("_", " ").split() if p]
        part_vecs = [wv[p] for p in parts if p in wv]
        if part_vecs:
            # media vettore del lemma
            lemma_vecs.append(np.mean(np.stack(part_vecs, axis=0), axis=0))
    base = mean_vec(lemma_vecs, dim)

    if not use_gloss:
        return base

    # 2. Vettori della glossa (definizione testuale)
    gloss_tokens = [normalize_token(t) for t in word_tokenize(syn.definition())]
    gloss_tokens = [t for t in gloss_tokens if t and t in wv]
    # converte la definizione in embedding medio
    gloss_vec = mean_vec([wv[t] for t in gloss_tokens], dim)

    # combinazione col vettore dei lemma
    out = base + gloss_weight * gloss_vec

    # 3. Vettori degli esempi (contesto extra)
    if use_examples:
        ex_text = " ".join(syn.examples())
        if ex_text.strip():
            ex_tokens = [normalize_token(t) for t in word_tokenize(ex_text)]
            ex_tokens = [t for t in ex_tokens if t and t in wv]
            # media vettore dell'esempio
            ex_vec = mean_vec([wv[t] for t in ex_tokens], dim)
            out = out + examples_weight * ex_vec

    return out

# -----------------------------
# 4) Context embedding (frase -> vettore)
#    costruisce il vettore del contesto della frase
# -----------------------------
def context_embedding(
    sentence: str,
    target_word: str,
    wv: KeyedVectors,
    remove_stopwords: bool = True,
) -> np.ndarray:
    """
    Crea un vettore che rappresenta il contesto della parola target
    Viene calcolato come media dei vettori delle parole nella frase,
    eccetto la parola target (e opzionalmente le stopwords).
    """
    dim = wv.vector_size
    sw = set(stopwords.words("english")) if remove_stopwords else set()

    tokens = [normalize_token(t) for t in word_tokenize(sentence)]
    tokens = [t for t in tokens if t]

    target = normalize_token(target_word)

    ctx_vecs = []
    for t in tokens:
        # esclude la parola target (altrimenti 'bara')
        if t == target:
            continue
        if remove_stopwords and t in sw:
            continue
        if t in wv:
            ctx_vecs.append(wv[t])

    # ritorna un vettore che rappresenta 'di cosa parla' il contesto
    return mean_vec(ctx_vecs, dim)

# -----------------------------
# 5) WSD: selezione del synset migliore
# -----------------------------
def disambiguate_wordnet_wsd(
    sentence: str,
    target_word: str,
    wv: KeyedVectors,
    pos: Optional[str] = None, # wn.NOUN / wn.VERB / wn.ADJ / wn.ADV oppure None
    use_gloss: bool = True,
    gloss_weight: float = 0.5,
    topk: int = 5,
) -> Tuple[Optional[Synset], List[Tuple[Synset, float]]]:
    """
    Confronta il vettore del contesto con i vettori di tutti i possibili significati (synsets).
    Il significato con la similarità coseno più alta vince.
    """
    # crea il vettore del contesto attuale
    ctx = context_embedding(sentence, target_word, wv)
    if norm(ctx) == 0:
        return None, []

    # prende i possibili significati (candidati) da WordNet
    candidates = wn.synsets(target_word, pos=pos) if pos else wn.synsets(target_word)
    if not candidates:
        return None, []

    scored = []
    # per ogni significato, calcola la similarità con la frase
    for syn in candidates:
        # calcolo del vettore del synset
        svec = synset_embedding(syn, wv, use_gloss=use_gloss, gloss_weight=gloss_weight)
        # calcolo cosine similarity tra contesto e synset
        score = cosine_sim(ctx, svec)
        scored.append((syn, score))

    # ordina e prende il migliore
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[0][0], scored[:topk]

# -----------------------------
# 6) Demo / Test
# -----------------------------
def run_demo(wv: KeyedVectors):
    tests = [
        ("I went to the bank to deposit money.", "bank", wn.NOUN),
        ("The river overflowed the bank after the storm.", "bank", wn.NOUN),
        ("He swung the bat and hit the ball.", "bat", wn.NOUN),
        ("A bat flew out of the cave at night.", "bat", wn.NOUN),
        ("They built a new plant to produce cars.", "plant", wn.NOUN),
        ("I watered the plant on the windowsill.", "plant", wn.NOUN),
    ]

    for sent, target, pos in tests:
        best, ranked = disambiguate_wordnet_wsd(
            sent, target, wv,
            pos=pos,
            use_gloss=True,
            gloss_weight=0.5,
            topk=5,
        )
        print("=" * 80)
        print("Sentence:", sent)
        print("Target:  ", target)
        if best is None:
            print("No decision (maybe OOV context).")
            continue
        print("\nBest synset:", best.name())
        print("Definition:", best.definition())

        print("\nTop candidates:")
        for syn, sc in ranked:
            print(f"  {syn.name():20s}  score={sc:.4f}  def={syn.definition()}")

# -----------------------------
# MAIN
# -----------------------------
if __name__ == "__main__":
    ensure_nltk()

    wv = load_pretrained_vectors("fasttext-wiki-news-subwords-300")

    run_demo(wv)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Caricamento del modello fasttext-wiki-news-subwords-300 in corso...
✅ Loaded pretrained vectors: fasttext-wiki-news-subwords-300
Vocab size: 999999
Vector dim: 300
Sentence: I went to the bank to deposit money.
Target:   bank

Best synset: savings_bank.n.02
Definition: a container (usually with a slot in the top) for keeping money at home

Top candidates:
  savings_bank.n.02     score=0.7971  def=a container (usually with a slot in the top) for keeping money at home
  depository_financial_institution.n.01  score=0.7594  def=a financial institution that accepts deposits and channels the money into lending activities
  bank.n.06             score=0.7229  def=the funds held by a gambling house or the dealer in some gambling games
  bank.n.09             score=0.7157  def=a building in which the business of banking transacted
  bank.n.01             score=0.7072  def=sloping land (especially the slope beside a body of water)
Sentence: The river overflowed the bank after the storm.
Target: 